# Text2SQL 思路复现：检索增强生成（RAG）

对每个测试问题，从训练集中检索文字最相似的 3-5 个 `NL -> SQL` 样例，并把它们补充到原始 Text2SQL prompt 末尾。模型、生成参数、prompt 主体风格，以及表结构数量（3 张）均与 `baseline.ipynb` 保持一致。

这里的检索器使用 TF-IDF 字符 n-gram 和余弦相似度：它本身就是最简洁、无需额外嵌入模型的 RAG 方案。若后续发现同义改写较多而字面重合少，再替换为向量嵌入检索。

In [ ]:
import csv
import json
from pathlib import Path

import paddle
from tqdm import tqdm
from paddlenlp.transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [ ]:
# 与 baseline 保持一致的模型和表结构规模。
MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
NUM_TABLES = 3
TOP_K = 5  # 可设为 3、4 或 5

# 本地目录为 dataset；AI Studio 原目录为 data 时自动兼容。
DATA_DIR = Path('dataset') if Path('dataset').exists() else Path('data')
TRAIN_PATH = DATA_DIR / 'train.json'
TEST_PATH = DATA_DIR / 'test.json'
TABLES_PATH = DATA_DIR / 'tables.sql'

In [ ]:
def load_json(path):
    with path.open('r', encoding='utf-8') as f:
        # strict=False 兼容训练问题中可能存在的原始换行符。
        return json.load(f, strict=False)


def load_table_schema(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return f.read()


def extract_table_schemas(sql_content, num_tables=NUM_TABLES):
    tables = []
    for table_def in sql_content.split(';'):
        if 'CREATE TABLE' in table_def:
            tables.append(table_def.strip() + ';')
        if len(tables) == num_tables:
            break
    return '\n\n'.join(tables)


train_data = load_json(TRAIN_PATH)
test_data = load_json(TEST_PATH)
table_schemas = extract_table_schemas(load_table_schema(TABLES_PATH))
assert all({'NL', 'SQL'} <= item.keys() for item in train_data)
assert all({'id', 'NL'} <= item.keys() for item in test_data)
assert table_schemas.count('CREATE TABLE') == NUM_TABLES
print(f'train={len(train_data)}, test={len(test_data)}, tables={NUM_TABLES}')

In [ ]:
# ponytail: TF-IDF 已归一化，linear_kernel 即余弦相似度；语义召回不足时再换嵌入模型。
def build_retriever(train_items):
    questions = [item['NL'] for item in train_items]
    vectorizer = TfidfVectorizer(
        analyzer='char',
        ngram_range=(2, 4),
        sublinear_tf=True,
        norm='l2',
    )
    return vectorizer, vectorizer.fit_transform(questions)


def retrieve_examples(nl_question, train_items, vectorizer, question_matrix, top_k=TOP_K):
    if not 3 <= top_k <= 5:
        raise ValueError('top_k must be between 3 and 5')
    scores = linear_kernel(vectorizer.transform([nl_question]), question_matrix).ravel()
    indices = scores.argsort()[-top_k:][::-1]
    return [
        {'NL': train_items[i]['NL'], 'SQL': train_items[i]['SQL'], 'score': float(scores[i])}
        for i in indices
    ]


vectorizer, question_matrix = build_retriever(train_data)
retrieved = retrieve_examples(test_data[0]['NL'], train_data, vectorizer, question_matrix)
assert len(retrieved) == TOP_K and all({'NL', 'SQL', 'score'} <= item.keys() for item in retrieved)
retrieved

In [ ]:
def format_retrieved_examples(examples):
    return '\n\n'.join(
        f'示例 {index}：\n问题：{example["NL"]}\n查询SQL：{example["SQL"]}'
        for index, example in enumerate(examples, start=1)
    )


def build_prompt(nl_question, table_schemas, retrieved_examples):
    if isinstance(table_schemas, list):
        table_schemas = '\n'.join(table_schemas)
    prompt = (
        '### 任务：将以下自然语言问题转换为 SQL，只输出 SQL 语句，不要任何解释和注释。\n'
        '### 格式示例：SELECT column1, column2 FROM table WHERE condition;\n\n'
        '问题：\n'
        f'```\n{nl_question}\n```\n\n'
        '表结构：\n'
        f'```\n{table_schemas or "无"}\n```\n\n'
        '相似训练示例（仅参考问题-SQL 映射，结合当前问题独立生成 SQL）：\n'
        f'{format_retrieved_examples(retrieved_examples)}'
    )
    return prompt


example_prompt = build_prompt(test_data[0]['NL'], table_schemas, retrieved)
assert '相似训练示例' in example_prompt and all(item['SQL'] in example_prompt for item in retrieved)
print(example_prompt)

In [ ]:
# 模型调用与 baseline 保持不变。
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype='float16')


def generate_sql_with_qwen(prompt):
    try:
        inputs = tokenizer(prompt, return_tensors='pd')
        outputs = model.generate(
            input_ids=inputs['input_ids'],
            max_new_tokens=512,
            temperature=0.1,
            top_p=0.7,
            repetition_penalty=1.1,
        )
        output_ids = outputs[0][0].tolist()
        sql = tokenizer.decode(output_ids, skip_special_tokens=True)
        if prompt in sql:
            sql = sql[len(prompt):].strip()
        return sql.rsplit('查询SQL:', 1)[-1].strip()
    except Exception as e:
        print(f'生成 SQL 时出错: {e}')
        return None

In [ ]:
results = []
for item in tqdm(test_data, desc='Processing', unit='sample'):
    nl_question = item['NL']
    examples = retrieve_examples(nl_question, train_data, vectorizer, question_matrix)
    prompt = build_prompt(nl_question, table_schemas, examples)
    results.append({'id': item['id'], 'pred_sql': generate_sql_with_qwen(prompt)})

with open('submission.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['id', 'pred_sql'])
    writer.writeheader()
    writer.writerows(results)

print(f'成功写入 submission.csv，共 {len(results)} 条记录。')